# Ingest drivers.json file
1. Read the file using spark dataframe reader API
2. Define and enforce schema (preserve the nested structure)
3. Add Metadata Columns 
    - Source File
    - Ingestion Timestamp
4. Write to bronze delta table   

#### Step 0 - Run configuration notebooks and set variables

In [0]:
dbutils.widgets.text("p_batch_id", "")

v_batch_id = dbutils.widgets.get("p_batch_id")
print(v_batch_id)

In [0]:
%run ../00_common/01_configuration

In [0]:
%run ../00_common/02_bronze_functions

In [0]:
source_file = f"{landing_folder_path}/{v_batch_id}/drivers.json"
table_name = f"{catalog_name}.{bronze_schema}.drivers"

#### Step 1 - Read the JSON file using the dataframe reader API

In [0]:
# Define the schema
from pyspark.sql.types import StructType, StructField, StringType, DateType

name_schema = StructType([
    StructField('givenName', StringType()),
    StructField('familyName', StringType())
])

drivers_schema = StructType([
    StructField('driverId', StringType()),
    StructField('name', name_schema),
    StructField('dateOfBirth', DateType()),
    StructField('nationality', StringType()),
    StructField('url', StringType())
])

In [0]:
# Read the data
drivers_df = (
    spark.read
       .format('json')
       .schema(drivers_schema)
       .option('mode', 'FAILFAST')
       .load(source_file)
)

In [0]:
# display(drivers_df)

#### Step 2 - Add Metadata Columns
- Source File
- Ingestion Timestamp

In [0]:
drivers_final_df = add_ingestion_metadata(drivers_df)

#### Step 3 - Write to bronze delta table

In [0]:
write_to_bronze (
    input_df = drivers_final_df,
    target_table = table_name,
    batch_id = v_batch_id
)

In [0]:
spark.table(table_name).display()